# Deep Learning — Module 2: Training & Optimization · Part 4
## Normalization Layers & Weight Initialization

> Every concept in three layers: **Intuition → Math → Code**

---

## Table of Contents

| Section | Topic |
|---------|-------|
| **1** | Why Normalization Matters — Internal Covariate Shift |
| **2** | Batch Normalization — Math + Forward + Backward |
| **3** | Batch Norm at Test Time — Running Statistics |
| **4** | Layer Normalization (Transformers) |
| **5** | Instance Normalization (Style Transfer) |
| **6** | Group Normalization |
| **7** | Comparison Table: Which Norm, When? |
| **8** | Why Initialization Matters |
| **9** | Zero Init & Symmetry Breaking Problem |
| **10** | Xavier / Glorot Initialization |
| **11** | He (Kaiming) Initialization |
| **12** | Orthogonal Initialization |
| **13** | PyTorch: Normalization + Init Demo |
| **14** | Master Interview Q&A Cheatsheet |


In [ ]:
# ── All imports ────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "axes.prop_cycle": plt.cycler(color=[
        "#4C72B0","#DD8452","#55A868","#C44E52","#8172B3","#937860"
    ])
})
print("Imports ready ✓")


## 1. Why Normalization Matters — Internal Covariate Shift

### Intuition: The Moving Target Problem
Imagine you are learning to catch a ball. Every time you reach for it, someone slightly changes the throwing speed, angle, and spin — so the ball arrives differently each time.

Your brain spends half its effort just re-adapting to the changing "input distribution" instead of learning to actually catch the ball.

This is exactly what happens to each layer in a deep network:
- Layer 2 feeds inputs to Layer 3
- But as Layer 2's weights update during training, the **distribution of its outputs shifts**
- Layer 3 must constantly re-adapt to this shifting input — this is **Internal Covariate Shift (ICS)**

The more layers you have, the worse this gets — each layer's shift compounds.

---

### The Formal Problem

At layer $l$, the pre-activation input is:
$$z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}$$

As $W^{(l-1)}$ (previous layer weights) change during training, the **distribution of $a^{(l-1)}$** changes.

Consequences:
1. **Slower learning** — each layer must chase a moving target distribution
2. **Gradient instability** — extreme input values push activations into saturation zones
3. **Sensitivity to initialization** — bad init + shifting distributions = dead neurons or exploding signals

### What Normalization Does
Normalization layers **standardize the inputs** to each layer (zero mean, unit variance) before passing them forward.

This means:
- Each layer always sees inputs from a **stable, predictable distribution**
- Gradients flow more smoothly (activations stay in linear regime)
- Training can use **larger learning rates** (3–10× larger with BatchNorm)
- Models are **less sensitive to initialization**

The key question: **what do you normalise over?** (batch, sequence length, channel, group)
→ This is exactly what distinguishes BatchNorm, LayerNorm, InstanceNorm, and GroupNorm.


In [ ]:
# Figure 1: Internal Covariate Shift — distribution shift across training
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# Simulate pre-activation distributions at layer 2 across training steps
class NoNormNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(20, 64)
        self.l2 = nn.Linear(64, 64)
    def forward(self, x):
        return self.l2(torch.relu(self.l1(x)))

class BNNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(20, 64)
        self.bn = nn.BatchNorm1d(64)
        self.l2 = nn.Linear(64, 64)
    def forward(self, x):
        return self.l2(self.bn(torch.relu(self.l1(x))))

net_no = NoNormNet(); net_bn = BNNet()
x_fixed = torch.randn(256, 20)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

snapshots = [0, 5, 20]
opts = [torch.optim.SGD(net_no.parameters(), lr=0.01),
        torch.optim.SGD(net_bn.parameters(), lr=0.01)]

for ax_col, step in enumerate(snapshots):
    # Train a few steps
    for opt, net in zip(opts, [net_no, net_bn]):
        for _ in range(step):
            opt.zero_grad()
            y_fake = net(x_fixed).sum()
            y_fake.backward()
            opt.step()

    # Collect layer-2 pre-activation distributions
    with torch.no_grad():
        acts_no = net_no.l2(torch.relu(net_no.l1(x_fixed))).numpy().flatten()
        acts_bn = net_bn.l2(net_bn.bn(torch.relu(net_bn.l1(x_fixed)))).numpy().flatten()

    for ax_row, (acts, title, col) in enumerate([
        (acts_no, f'No Norm — step {step}', '#C44E52'),
        (acts_bn, f'BatchNorm — step {step}', '#55A868'),
    ]):
        ax = axes[ax_row][ax_col]
        ax.hist(acts, bins=50, color=col, alpha=0.75, density=True)
        ax.set_xlim(-6, 6)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel('Pre-activation value')
        ax.set_ylabel('Density')
        ax.text(0.97, 0.92, f'μ={acts.mean():.2f}
σ={acts.std():.2f}',
                transform=ax.transAxes, ha='right', fontsize=9, color=col,
                bbox=dict(boxstyle='round', facecolor='white', edgecolor=col, alpha=0.8))

plt.suptitle('Figure 1: Internal Covariate Shift — Distribution Shifts Without Normalization',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Batch Normalization — Math, Forward Pass & Learning Parameters

### Intuition: Standardize Each Feature Across the Batch
BatchNorm looks at the current mini-batch and asks:
*"What is the mean and variance of each feature across all examples in this batch?"*

It then normalises each feature to have zero mean and unit variance — then applies learnable scale $\gamma$ and shift $\beta$ to restore the network's expressiveness.

---

### Forward Pass (4 steps)

Given a mini-batch $\mathcal{B} = \{z_1, z_2, ..., z_B\}$ of pre-activations for one feature:

**Step 1: Batch mean**
$$\mu_{\mathcal{B}} = \frac{1}{B}\sum_{i=1}^{B} z_i$$

**Step 2: Batch variance**
$$\sigma^2_{\mathcal{B}} = \frac{1}{B}\sum_{i=1}^{B}(z_i - \mu_{\mathcal{B}})^2$$

**Step 3: Normalize**
$$\hat{z}_i = \frac{z_i - \mu_{\mathcal{B}}}{\sqrt{\sigma^2_{\mathcal{B}} + \epsilon}}$$

**Step 4: Scale & Shift (learnable)**
$$y_i = \gamma \hat{z}_i + \beta$$

- $\gamma, \beta$ are **learned by backprop** — one per feature/channel
- $\epsilon \approx 10^{-5}$: numerical stability
- If $\gamma = \sqrt{\sigma^2}$ and $\beta = \mu$: identity (can undo normalization)

### Where BatchNorm is Applied
```
Linear / Conv → BatchNorm → Activation (ReLU)
```
Always **before** the activation, **after** the linear transform.

### Why $\gamma$ and $\beta$?
Without them, every layer would be forced to operate in the linear regime of the activation (near 0).
$\gamma$ and $\beta$ let the network **learn the optimal scale and shift** for each feature — it can choose to undo normalization entirely if that's what the task requires.

### Benefits of BatchNorm
| Benefit | Why |
|---------|-----|
| Faster training | Larger LRs possible (10× in some cases) |
| Less sensitive to init | Normalisation prevents init-caused explosions |
| Slight regularization | Batch statistics add noise (like dropout) |
| Stabilizes gradient flow | Prevents extreme activations entering saturation |


In [ ]:
# Figure 2: BatchNorm Forward Pass — step by step + learnable params
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)

# ── Manual BatchNorm forward (numpy) ──────────────────────────────
B, C = 8, 1     # batch of 8, single feature for clarity
z = np.array([2.1, -0.5, 3.2, 0.8, -1.2, 4.5, 1.1, 0.3])  # pre-activations

eps = 1e-5
mu  = z.mean()
var = z.var()
z_hat = (z - mu) / np.sqrt(var + eps)
gamma, beta = 1.5, 0.3   # learnable params
y = gamma * z_hat + beta

print("══ BatchNorm Forward — Manual ══════════════════════════")
print(f"  Input z:    {z}")
print(f"  μ_B:        {mu:.4f}")
print(f"  σ²_B:       {var:.4f}")
print(f"  ẑ (norm):   {z_hat.round(4)}")
print(f"  y = γẑ + β: {y.round(4)}")
print(f"  γ={gamma}, β={beta}")
print()

# ── Verify with PyTorch ───────────────────────────────────────────
z_t = torch.tensor(z, dtype=torch.float32).unsqueeze(1)   # (8,1)
bn  = nn.BatchNorm1d(1, eps=eps, momentum=0.1)
bn.weight.data.fill_(gamma)
bn.bias.data.fill_(beta)
bn.train()
y_t = bn(z_t).detach().numpy().flatten()
print(f"  PyTorch BN: {y_t.round(4)}")
print(f"  Max diff (manual vs torch): {np.abs(y - y_t).max():.2e}")
print()

# ── Visualisation ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

stages = [
    (z,     'Input z
(raw pre-activations)',    '#C44E52'),
    (z-mu,  'z − μ
(centered)',                 '#DD8452'),
    (z_hat, 'ẑ = (z−μ)/σ
(normalized)',         '#4C72B0'),
    (y,     'y = γẑ + β
(scaled & shifted)',     '#55A868'),
]
for ax, (vals, title, col) in zip(axes, stages):
    ax.bar(range(len(vals)), vals, color=col, alpha=0.85)
    ax.axhline(0, color='grey', lw=1, ls='--')
    ax.axhline(vals.mean(), color='black', lw=1.5, ls='-', label=f'μ={vals.mean():.2f}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Sample in batch')
    ax.set_xticks(range(len(vals)))
    ax.text(0.97, 0.94, f'μ={vals.mean():.2f}
σ={vals.std():.2f}',
            transform=ax.transAxes, ha='right', fontsize=9, color=col,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=col, alpha=0.7))

plt.suptitle('Figure 2: BatchNorm Forward Pass — 4 Steps', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Impact on training speed ──────────────────────────────────────
torch.manual_seed(0)

def make_deep(use_bn):
    layers = []
    for i in range(8):
        layers.append(nn.Linear(64, 64))
        if use_bn: layers.append(nn.BatchNorm1d(64))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(64, 1))
    return nn.Sequential(*layers)

X = torch.randn(256, 64)
y_target = torch.randn(256, 1)

for use_bn, label, col in [(False,'No BN','#C44E52'),(True,'With BN','#55A868')]:
    net = make_deep(use_bn)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    hist = []
    for _ in range(200):
        net.train(); opt.zero_grad()
        l = nn.MSELoss()(net(X), y_target)
        if not torch.isnan(l): l.backward(); opt.step()
        net.eval()
        with torch.no_grad(): hist.append(nn.MSELoss()(net(X), y_target).item())

fig2, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist, color=col, lw=2, label=label)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Figure 2b: Training Speed — 8-Layer Net with vs without BN')
ax.legend(); plt.tight_layout(); plt.show()


## 3. Batch Normalization at Test Time — Running Statistics

### The Test-Time Problem
At training time, BatchNorm uses the **current mini-batch** statistics ($\mu_{\mathcal{B}}, \sigma^2_{\mathcal{B}}$).

At test time:
- We often predict **one sample at a time** → no "batch" to compute statistics over
- Batch of 1 has $\mu = z$ and $\sigma = 0$ → normalised output = 0 always
- Even with batch > 1: test batches are different from training batches

**Solution: Maintain running estimates** of mean and variance during training.

---

### Running Statistics (Exponential Moving Average)

After each training batch, update running estimates:
$$\mu_{\text{run}} \leftarrow (1 - m)\mu_{\text{run}} + m\cdot\mu_{\mathcal{B}}$$
$$\sigma^2_{\text{run}} \leftarrow (1-m)\sigma^2_{\text{run}} + m\cdot\sigma^2_{\mathcal{B}}$$

- $m$ = momentum (PyTorch default: 0.1) — how much to update from current batch
- After training, $\mu_{\text{run}}$ and $\sigma^2_{\text{run}}$ approximate the **population statistics**

### Test-time Normalisation
$$\hat{z}_i = \frac{z_i - \mu_{\text{run}}}{\sqrt{\sigma^2_{\text{run}} + \epsilon}}$$
$$y_i = \gamma\hat{z}_i + \beta$$

Same $\gamma, \beta$ learned during training — no change.
Only the statistics source changes: **batch stats → running stats**.

### Why `model.train()` / `model.eval()` Switches These
```python
model.train()   # BN uses batch mean/var; updates running stats
model.eval()    # BN uses running mean/var; does NOT update
```

This is the second reason (besides Dropout) to always call `model.eval()` before inference.

### The Batch Size Constraint
BatchNorm requires a sufficiently large batch to get stable statistics.
- Batch size < 8: statistics too noisy → training unstable
- Typical minimum: 16–32 per GPU
- **This is why BatchNorm fails with very small batches** → use LayerNorm or GroupNorm instead

#### Interview Questions: BatchNorm Test Time
> **Q: Why does BatchNorm behave differently at train vs test time?**
> A: Training: uses current mini-batch statistics (noisy but up-to-date). Test: uses running statistics accumulated over training (stable population estimates). `model.eval()` switches between these modes.

> **Q: What happens if you forget model.eval() with a BatchNorm model?**
> A: BatchNorm continues using mini-batch statistics from whatever you pass in as a "batch". For single samples, this is degenerate. For small test batches, the statistics are noisy and inconsistent with training, causing unpredictable outputs.

> **Q: What is the momentum parameter in BatchNorm?**
> A: Controls how fast running statistics update. Higher momentum → faster update (more weight to current batch). PyTorch default is 0.1 (10% from new batch each step). Not to be confused with optimizer momentum.


In [ ]:
# Figure 3: Running Statistics — convergence to population stats
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(0)

# True population: N(3.0, 2.0²)
true_mu, true_std = 3.0, 2.0
n_steps = 200
batch_size = 32

bn = nn.BatchNorm1d(1, momentum=0.1)
bn.train()

running_means, running_vars = [], []
batch_means_hist = []

for _ in range(n_steps):
    x_batch = torch.randn(batch_size, 1) * true_std + true_mu
    _ = bn(x_batch)   # triggers running stat update
    running_means.append(bn.running_mean.item())
    running_vars.append(bn.running_var.item())
    batch_means_hist.append(x_batch.mean().item())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
steps = range(n_steps)

# ── Plot 1: Running mean convergence ──
axes[0].plot(steps, running_means,    color='#4C72B0', lw=2, label='Running mean')
axes[0].plot(steps, batch_means_hist, color='#DD8452', lw=1, alpha=0.4, label='Batch mean (noisy)')
axes[0].axhline(true_mu, color='#C44E52', ls='--', lw=2, label=f'True μ={true_mu}')
axes[0].set_xlabel('Training step'); axes[0].set_ylabel('Mean estimate')
axes[0].set_title('Running Mean → True Population Mean'); axes[0].legend()

# ── Plot 2: Running var convergence ──
running_stds = np.sqrt(np.array(running_vars))
axes[1].plot(steps, running_stds,    color='#55A868', lw=2, label='Running std')
axes[1].axhline(true_std, color='#C44E52', ls='--', lw=2, label=f'True σ={true_std}')
axes[1].set_xlabel('Training step'); axes[1].set_ylabel('Std estimate')
axes[1].set_title('Running Std → True Population Std'); axes[1].legend()

# ── Plot 3: Effect of momentum on convergence rate ──
for mom, col in [(0.01,'#4C72B0'),(0.1,'#55A868'),(0.5,'#DD8452')]:
    bn_m = nn.BatchNorm1d(1, momentum=mom); bn_m.train()
    rm = []
    for _ in range(n_steps):
        xb = torch.randn(32,1)*true_std + true_mu
        bn_m(xb)
        rm.append(bn_m.running_mean.item())
    axes[2].plot(steps, rm, color=col, lw=2, label=f'momentum={mom}')
axes[2].axhline(true_mu, color='#C44E52', ls='--', lw=2, label=f'True μ={true_mu}')
axes[2].set_xlabel('Training step'); axes[2].set_ylabel('Running mean')
axes[2].set_title('Effect of BN momentum on Convergence Rate'); axes[2].legend()

plt.suptitle('Figure 3: BatchNorm Running Statistics (Train→Test)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 4. Layer Normalization — The Transformer Standard

### The BatchNorm Problem for Transformers
BatchNorm normalises **across the batch dimension** (per feature):
- Requires batch size ≥ 16 for stable statistics
- Fails for variable-length sequences (batch padding corrupts stats)
- Doesn't work for single-sample inference (batch=1)
- Auto-regressive decoding at batch=1 → can't use BatchNorm

**Layer Normalization** normalises **across the feature dimension** (per sample):

$$\hat{z}_{i} = \frac{z_i - \mu_i^{(l)}}{\sqrt{(\sigma_i^{(l)})^2 + \epsilon}} \cdot \gamma + \beta$$

Where $\mu_i^{(l)}$ and $\sigma_i^{(l)}$ are computed over **all features for sample $i$**:

$$\mu_i^{(l)} = \frac{1}{H}\sum_{j=1}^{H}z_{ij}, \qquad \sigma_i^{(l)} = \sqrt{\frac{1}{H}\sum_{j=1}^{H}(z_{ij}-\mu_i^{(l)})^2 + \epsilon}$$

- $H$ = number of features (hidden dimension)
- Statistics computed **per sample, across all features**

### Key Difference: What Is Normalised Over

| Method | Normalises over | Statistics are per... |
|--------|----------------|----------------------|
| BatchNorm | Batch (N) dimension | Feature/channel |
| LayerNorm | Feature (H) dimension | Sample |

### Why LayerNorm Works for Transformers
- **No batch dependency**: each sample normalised independently → works at batch=1
- **Variable-length sequences**: normalise each token independently → no padding issue
- **Auto-regressive decoding**: single-token inference at test time — fine
- **Same behaviour at train and test**: no running statistics needed, no `model.eval()` difference for LN

### Where LayerNorm is Applied in Transformers (Pre-LN)
```
x → LayerNorm → Self-Attention → + residual → LayerNorm → FFN → + residual
```
Modern transformers use **Pre-LayerNorm** (before sublayer) for stable gradients.

#### Interview Questions: LayerNorm
> **Q: Why does LayerNorm work at batch size 1 but BatchNorm doesn't?**
> A: LayerNorm normalises over the feature dimension (per sample) — statistics depend only on the current sample. BatchNorm normalises over the batch dimension — with batch=1, you have one sample, so variance=0 and normalisation is degenerate.

> **Q: Why do transformers use LayerNorm instead of BatchNorm?**
> A: (1) Sequences are variable-length — batch statistics with padding are corrupted. (2) Auto-regressive generation uses batch=1. (3) LayerNorm trains and infers identically (no running stat switch). (4) Empirically LayerNorm works better for attention-based models.


In [ ]:
# Figure 4: LayerNorm — what it normalises + comparison with BatchNorm
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)

# (N=4 samples, H=6 features)
z = torch.tensor([
    [2.1, -0.5,  3.2,  0.8, -1.2,  4.5],
    [0.3,  1.1, -2.0,  0.5,  0.9,  1.8],
    [5.0, -1.5,  2.3, -0.2,  3.1,  0.7],
    [1.2,  0.4,  0.9,  2.2, -0.8,  1.5],
], dtype=torch.float32)   # shape (4, 6)

# BatchNorm: normalise over N for each feature
bn = nn.BatchNorm1d(6, eps=1e-5, affine=False)
z_bn = bn(z).detach().numpy()

# LayerNorm: normalise over H for each sample
ln = nn.LayerNorm(6, eps=1e-5, elementwise_affine=False)
z_ln = ln(z).detach().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Plot 1: Heatmap — show what gets normalised ──
im0 = axes[0].imshow(z.numpy(), cmap='RdYlGn', aspect='auto', vmin=-2, vmax=5)
axes[0].set_title('Input Z (4 samples × 6 features)')
axes[0].set_xlabel('Feature index'); axes[0].set_ylabel('Sample index')
plt.colorbar(im0, ax=axes[0])
# Annotate arrows
axes[0].annotate('BatchNorm
normalises
down (per feature)', xy=(0,0),
                 xytext=(6.5, 1.5), fontsize=9, color='#4C72B0',
                 arrowprops=dict(arrowstyle='->', color='#4C72B0'))
axes[0].annotate('LayerNorm
normalises
across (per sample)', xy=(3,0),
                 xytext=(1.5, 5.0), fontsize=9, color='#C44E52',
                 arrowprops=dict(arrowstyle='->', color='#C44E52'))

im1 = axes[1].imshow(z_bn, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)
axes[1].set_title('After BatchNorm
(each col: μ≈0, σ≈1)')
axes[1].set_xlabel('Feature index'); axes[1].set_ylabel('Sample index')
plt.colorbar(im1, ax=axes[1])
for j in range(6):
    col_vals = z_bn[:, j]
    axes[1].text(j, -0.6, f'σ={col_vals.std():.2f}', ha='center', fontsize=8, color='#4C72B0')

im2 = axes[2].imshow(z_ln, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)
axes[2].set_title('After LayerNorm
(each row: μ≈0, σ≈1)')
axes[2].set_xlabel('Feature index'); axes[2].set_ylabel('Sample index')
plt.colorbar(im2, ax=axes[2])
for i in range(4):
    row_vals = z_ln[i, :]
    axes[2].text(6.1, i, f'σ={row_vals.std():.2f}', ha='left', fontsize=8, color='#C44E52')

plt.suptitle('Figure 4: BatchNorm vs LayerNorm — What Gets Normalised',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ── Stats verification ──
print("After BatchNorm (column-wise mean/std):")
print("  μ per feature:", z_bn.mean(axis=0).round(2))
print("  σ per feature:", z_bn.std(axis=0).round(2))
print()
print("After LayerNorm (row-wise mean/std):")
print("  μ per sample:", z_ln.mean(axis=1).round(2))
print("  σ per sample:", z_ln.std(axis=1).round(2))


## 5. Instance Normalization — Style Transfer

### Intuition
Used primarily in **style transfer** and **image generation**.

BatchNorm: normalise per feature across the batch
InstanceNorm: normalise **each sample × each channel independently**

For an image input of shape $(N, C, H, W)$:
- BatchNorm: normalise over $(N, H, W)$ for each channel $C$
- InstanceNorm: normalise over $(H, W)$ for each sample & channel pair

$$\mu_{nc} = \frac{1}{HW}\sum_{h,w} z_{nchw}, \qquad  \hat{z}_{nchw} = \frac{z_{nchw} - \mu_{nc}}{\sqrt{\sigma^2_{nc} + \epsilon}}$$

### Why Style Transfer
Style transfer separates **content** (spatial structure) from **style** (channel statistics).
InstanceNorm removes instance-specific contrast info within each channel → style can be re-applied without the original contrast interfering.

---

## 6. Group Normalization — BatchNorm for Small Batches

### Problem
BatchNorm fails with small batch sizes (< 8):
- Statistics over too few samples → too noisy
- CV tasks on segmentation/object detection use batch size 1-2 per GPU (large images)

### Solution: Group Normalization (Wu & He, 2018)
Divide channels into $G$ groups. Normalise over the spatial dimensions **and** within each group:

$$\mu_{ng} = \frac{1}{(C/G) \cdot H \cdot W}\sum_{c\in g, h, w} z_{nchw}$$

- $G = 1$: reduces to LayerNorm (across all channels + spatial)
- $G = C$: reduces to InstanceNorm (each channel independent)
- $G = 8$ or $G = 32$: typical GroupNorm

### When to Use GroupNorm
- Object detection (Mask R-CNN, Detectron2) with batch size 1-2
- Large resolution images that don't fit many samples per GPU
- Any situation where BatchNorm's batch-size constraint is a bottleneck


In [ ]:
# Figure 5: InstanceNorm + GroupNorm — what they normalise
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(0)

# Image tensor: (N=2, C=4, H=4, W=4)
x = torch.randn(2, 4, 4, 4) * 3 + 1   # non-zero mean

inst_norm  = nn.InstanceNorm2d(4, affine=False)
group_norm = nn.GroupNorm(num_groups=2, num_channels=4, affine=False)
batch_norm = nn.BatchNorm2d(4, affine=False)

x_in  = inst_norm(x).detach()
x_gn  = group_norm(x).detach()
x_bn  = batch_norm(x).detach()

fig, axes = plt.subplots(2, 4, figsize=(15, 6))

titles = ['Original', 'BatchNorm2d', 'InstanceNorm2d', 'GroupNorm(G=2)']
tensors = [x, x_bn, x_in, x_gn]

for row in range(2):     # two samples
    for col, (title, t) in enumerate(zip(titles, tensors)):
        # Show first channel of each sample
        img = t[row, 0].numpy()
        im = axes[row][col].imshow(img, cmap='RdYlGn', vmin=-2, vmax=2)
        axes[row][col].set_title(f'{title}
Sample {row}, Ch 0', fontsize=9)
        axes[row][col].set_xlabel(f'μ={img.mean():.2f}, σ={img.std():.2f}', fontsize=8)
        axes[row][col].set_xticks([]); axes[row][col].set_yticks([])
        plt.colorbar(im, ax=axes[row][col])

plt.suptitle('Figure 5: Instance Norm vs Group Norm vs Batch Norm on 2D Images',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Print stats to confirm
print("BatchNorm   — mean per channel (across both samples):")
for c in range(4):
    print(f"  Ch {c}: μ={x_bn[:,c].mean():.4f}, σ={x_bn[:,c].std():.4f}")
print()
print("InstanceNorm — mean per sample per channel:")
for n in range(2):
    for c in range(4):
        print(f"  Sample {n}, Ch {c}: μ={x_in[n,c].mean():.4f}, σ={x_in[n,c].std():.4f}")


## 7. Comparison Table — Which Normalization, When?

### Normalisation Axes Summary

| Method | Normalises over | Shape input | Use case |
|--------|----------------|-------------|----------|
| **BatchNorm** | Batch (N) + spatial | (N, C, H, W) or (N, C) | CNNs, MLPs (batch≥16) |
| **LayerNorm** | Feature (C+H+W) | (N, C) or (N, L, C) | Transformers, RNNs |
| **InstanceNorm** | Spatial (H×W) per channel | (N, C, H, W) | Style transfer, GANs |
| **GroupNorm** | Group(C/G) + spatial | (N, C, H, W) | Object detection, small batches |

### Decision Guide
```
Are you training a Transformer or NLP model?
  → LayerNorm (always)

Are you training a CNN with batch size ≥ 16?
  → BatchNorm (best empirical performance)

Are you training a CNN with batch size < 8?
  → GroupNorm (G=32 is a good default)

Are you doing style transfer or image generation (GAN)?
  → InstanceNorm

Are you doing object detection (e.g., Mask R-CNN)?
  → GroupNorm (large images → small batch per GPU)
```

### Practical PyTorch API

```python
# BatchNorm
nn.BatchNorm1d(num_features)          # for (N, C) tensors
nn.BatchNorm2d(num_features)          # for (N, C, H, W) tensors

# LayerNorm
nn.LayerNorm(normalized_shape)        # e.g., [512] or [512, 768]

# InstanceNorm
nn.InstanceNorm2d(num_features)

# GroupNorm
nn.GroupNorm(num_groups=32, num_channels=C)
```

### Key Properties

| Property | BatchNorm | LayerNorm | InstanceNorm | GroupNorm |
|----------|-----------|-----------|--------------|-----------|
| Needs large batch? | ✅ Yes | ❌ No | ❌ No | ❌ No |
| Same at train/test? | ❌ No | ✅ Yes | ✅ Yes | ✅ Yes |
| Works at batch=1? | ❌ No | ✅ Yes | ✅ Yes | ✅ Yes |
| Learnable γ, β? | ✅ Yes | ✅ Yes | Optional | ✅ Yes |
| Sequence-friendly? | ❌ No | ✅ Yes | ❌ No | ❌ No |


In [ ]:
# Figure 6: All normalizations — sensitivity to batch size
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)

batch_sizes = [1, 2, 4, 8, 16, 32, 64]
C, H, W = 32, 8, 8

def test_norm_stability(norm_class, batch_sizes):
    stabilities = []
    for B in batch_sizes:
        x = torch.randn(B, C, H, W) * 3 + 2
        try:
            if norm_class == nn.BatchNorm2d:
                n = nn.BatchNorm2d(C, affine=False)
                n.train()
            elif norm_class == nn.LayerNorm:
                n = nn.LayerNorm([C, H, W], elementwise_affine=False)
            elif norm_class == nn.InstanceNorm2d:
                n = nn.InstanceNorm2d(C, affine=False)
            elif norm_class == nn.GroupNorm:
                n = nn.GroupNorm(num_groups=min(8, C), num_channels=C, affine=False)
            out = n(x).detach()
            stability = 1 - abs(out.std().item() - 1.0)   # how close to σ=1
            stabilities.append(min(max(stability, 0), 1))
        except Exception:
            stabilities.append(0)
    return stabilities

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

norms = [
    (nn.BatchNorm2d,    'BatchNorm2d', '#4C72B0'),
    (nn.LayerNorm,      'LayerNorm',   '#55A868'),
    (nn.InstanceNorm2d, 'InstanceNorm','#DD8452'),
    (nn.GroupNorm,      'GroupNorm',   '#8172B3'),
]
for norm_cls, name, col in norms:
    stab = test_norm_stability(norm_cls, batch_sizes)
    axes[0].plot(batch_sizes, stab, 'o-', lw=2.5, color=col, label=name, ms=8)

axes[0].set_xlabel('Batch size'); axes[0].set_ylabel('Output std ≈ 1? (higher=better)')
axes[0].set_title('Normalization Stability vs Batch Size')
axes[0].legend(); axes[0].set_xscale('log')
axes[0].axhline(0.95, color='grey', ls=':', lw=1)

# ── Plot 2: schematic diagram ──
axes[1].axis('off')
axes[1].set_xlim(0, 10); axes[1].set_ylim(0, 10)
axes[1].set_title('What Each Norm Normalises (N×C×H×W tensor)', fontsize=11, fontweight='bold')

labels = ['BatchNorm
(per channel,
across N+H+W)',
          'LayerNorm
(per sample,
across C+H+W)',
          'InstanceNorm
(per sample×ch,
across H+W)',
          'GroupNorm
(per group,
across H+W)']
colors_box = ['#4C72B0','#55A868','#DD8452','#8172B3']
x_positions = [1.2, 3.7, 6.2, 8.7]

for x, lbl, col in zip(x_positions, labels, colors_box):
    rect = FancyBboxPatch((x-1.0, 2.0), 2.0, 6.0,
                          boxstyle="round,pad=0.1",
                          facecolor=col, edgecolor='white', lw=2, alpha=0.25)
    axes[1].add_patch(rect)
    axes[1].text(x, 5.0, lbl, ha='center', va='center',
                 color=col if col != '#DD8452' else '#b85a00',
                 fontsize=8.5, fontweight='bold')

# Axes labels
axes[1].text(0.1, 8.5, 'N', fontsize=11, color='grey')
axes[1].text(0.1, 6.5, 'C', fontsize=11, color='grey')
axes[1].text(0.1, 4.5, 'H', fontsize=11, color='grey')
axes[1].text(0.1, 2.5, 'W', fontsize=11, color='grey')
axes[1].plot([0.5]*4, [8.5, 6.5, 4.5, 2.5], 'o', color='grey', ms=6)

plt.suptitle('Figure 6: Normalization Method Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---

## 8. Weight Initialization — Why It Matters

### Intuition: Starting in the Right Neighbourhood
Think of training as finding the lowest point in a huge landscape.
If you start from a random cliff edge (bad init), you may:
- Fall off (exploding gradients → NaN)
- Get stuck in a bad valley from the start (vanishing gradients → nothing learns)

Good initialization puts you in a sensible starting valley — not the best place, but a place where gradient flow works and training can proceed.

---

### What Can Go Wrong

**Too large weights:**
- Large pre-activations → activations saturate (sigmoid, tanh → flat gradient = 0)
- Gradients vanish before reaching early layers
- Loss = NaN in a few steps

**Too small weights:**
- Pre-activations near 0 → activations near 0
- Outputs are effectively 0 → predictions useless
- Gradients are tiny → training stalls

**All same weights (zero/constant init):**
- Symmetry breaking problem — all neurons compute identical gradients
- Training never differentiates them → model collapses to 1 effective neuron

### The Signal Propagation Ideal
For stable gradients, we want activation variance to be **preserved** through layers:

$$\text{Var}(a^{(l)}) \approx \text{Var}(a^{(l-1)}) \quad \text{for all layers } l$$

Both Xavier (for linear/tanh) and He (for ReLU) initializations are derived from this requirement.


In [ ]:
# Figure 7: Bad initialization — variance explosion and collapse
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)

n_layers = 10
n_units  = 128
N_samples = 512

x_in = torch.randn(N_samples, n_units)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, std_val, title, col in [
    (axes[0], 0.01,  'Too small weights (std=0.01)
→ Signal collapses', '#C44E52'),
    (axes[1], 1.0,   'Too large weights (std=1.0)
→ Signal explodes',   '#DD8452'),
    (axes[2], None,  'He init (std=√2/n)
→ Signal preserved',          '#55A868'),
]:
    var_per_layer = []
    x = x_in.clone()
    for l in range(n_layers):
        if std_val is not None:
            W = torch.randn(n_units, n_units) * std_val
        else:
            W = torch.randn(n_units, n_units) * np.sqrt(2.0 / n_units)
        x = torch.relu(x @ W.T)
        var_per_layer.append(x.var().item())

    ax.plot(range(1, n_layers+1), var_per_layer, 'o-', lw=2.5, color=col, ms=8)
    ax.set_xlabel('Layer depth'); ax.set_ylabel('Activation variance')
    ax.set_title(title)
    ax.axhline(1.0, color='grey', ls='--', lw=1.5, label='Ideal variance=1')
    ax.legend(fontsize=9)
    ax.set_yscale('symlog')

plt.suptitle('Figure 7: Signal Propagation — Effect of Initialization', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 9. Zero Initialization & The Symmetry Breaking Problem

### What Happens with Zero Init?

If all weights are initialised to 0:
$$W^{(l)} = 0 \implies z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)} = b^{(l)}$$

All neurons in layer $l$ compute **identical** pre-activations.
→ Identical activations → identical gradients → identical weight updates.

**After every training step, all neurons in a layer remain identical.**

No matter how long you train, the network behaves as if it had only 1 neuron per layer. This is the **symmetry breaking problem**.

### Constant Initialization is Equally Bad
Initialising all weights to the same non-zero constant has the same issue — all neurons remain symmetric.

### What to Init to Zero (and Why)
| Parameter | Init to Zero? | Reason |
|-----------|-------------|--------|
| **Weights $W$** | ❌ Never | Symmetry breaking |
| **Biases $b$** | ✅ Common | Biases don't have symmetry problem — OK to start at 0 |
| **BatchNorm γ** | ❌ (use 1) | Zero scale kills all output |
| **BatchNorm β** | ✅ OK | Zero shift is fine |
| **Output layer bias** | ✅ Or class-frequency log | Neutral starting prediction |

### The Random Init Solution
We need:
1. **Random** — break symmetry between neurons
2. **Small enough** — avoid saturation
3. **Large enough** — avoid signal collapse
4. **Scaled to layer width** — this is what Xavier and He solve precisely


In [ ]:
# Figure 8: Symmetry breaking — zero init vs random init
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(0)

# 2-layer net, 4 hidden neurons — watch them differentiate or not
class SmallNet(nn.Module):
    def __init__(self, zero_init):
        super().__init__()
        self.l1 = nn.Linear(2, 4, bias=False)
        self.l2 = nn.Linear(4, 1, bias=False)
        if zero_init:
            nn.init.zeros_(self.l1.weight)
            nn.init.zeros_(self.l2.weight)
        else:
            nn.init.normal_(self.l1.weight, std=0.5)
            nn.init.normal_(self.l2.weight, std=0.5)
    def forward(self, x):
        return self.l2(torch.relu(self.l1(x)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, zero_init, title in [
    (axes[0], True,  'Zero Initialization — All neurons stay identical (symmetry NOT broken)'),
    (axes[1], False, 'Random Initialization — Neurons differentiate (symmetry broken)'),
]:
    net = SmallNet(zero_init)
    opt = torch.optim.SGD(net.parameters(), lr=0.05)
    x   = torch.randn(32, 2)
    y   = (x[:,0] > 0).float().unsqueeze(1)

    weight_traces = [[] for _ in range(4)]  # track each hidden neuron's incoming weight norm

    for step in range(100):
        opt.zero_grad()
        nn.BCEWithLogitsLoss()(net(x), y).backward()
        opt.step()
        for i in range(4):
            weight_traces[i].append(net.l1.weight[i].norm().item())

    colors_w = ['#4C72B0','#DD8452','#55A868','#C44E52']
    for i, (trace, col) in enumerate(zip(weight_traces, colors_w)):
        ax.plot(trace, color=col, lw=2, label=f'Neuron {i}')
    ax.set_xlabel('Training step'); ax.set_ylabel('||w_i|| (weight norm)')
    ax.set_title(title, fontsize=9); ax.legend(fontsize=9)

    if zero_init:
        ax.text(50, max(weight_traces[0])*0.9,
                'All neurons identical — 1 effective neuron!',
                ha='center', fontsize=9, color='#C44E52',
                bbox=dict(boxstyle='round', facecolor='#fff0f0', edgecolor='#C44E52'))

plt.suptitle('Figure 8: Symmetry Breaking — Zero vs Random Init', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 10. Xavier / Glorot Initialization

### Goal: Preserve Signal Variance Through Linear Layers
Glorot & Bengio (2010) derived the optimal initialization for **linear layers with symmetric activations** (tanh, sigmoid):

For a layer $z = Wx$ where $x \in \mathbb{R}^{n_{\text{in}}}$, we want:

$$\text{Var}(z_j) = \text{Var}(x_i) \quad \forall i, j$$

If weights $w_{ij}$ are i.i.d. with mean 0 and variance $\sigma^2_W$, and inputs $x_i$ are i.i.d.:

$$\text{Var}(z_j) = n_{\text{in}} \cdot \sigma^2_W \cdot \text{Var}(x_i)$$

For variance preservation: $n_{\text{in}} \cdot \sigma^2_W = 1 \implies \sigma^2_W = \frac{1}{n_{\text{in}}}$

But we also want preservation in the backward pass: $\sigma^2_W = \frac{1}{n_{\text{out}}}$

**Glorot compromise:** use the harmonic mean:

$$\sigma^2_W = \frac{2}{n_{\text{in}} + n_{\text{out}}}$$

### Two Forms
**Xavier Uniform** (original paper):
$$W \sim \mathcal{U}\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}},\; +\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)$$

**Xavier Normal:**
$$W \sim \mathcal{N}\left(0,\; \frac{2}{n_{\text{in}} + n_{\text{out}}}\right)$$

### PyTorch
```python
nn.init.xavier_uniform_(layer.weight)     # uniform version
nn.init.xavier_normal_(layer.weight)      # normal version
```
PyTorch default for `nn.Linear` is **Kaiming (He) uniform**, NOT Xavier.

### When to Use Xavier
- Tanh or Sigmoid activation functions
- Linear output layers
- Embeddings

**Not ideal for ReLU** — ReLU kills half the neurons, so effective fan-in is $n_{\text{in}}/2$, requiring a factor of 2 correction → that's He initialization.


In [ ]:
# Figure 9: Xavier Init — variance preservation through tanh layers
import torch, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)

n_layers = 15
n_units  = 256
N_samp   = 1024
x0 = torch.randn(N_samp, n_units)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Compare inits with tanh activation
strategies = [
    ('Normal std=0.01',  lambda ni, no: torch.randn(no,ni)*0.01,            '#C44E52'),
    ('Xavier uniform',   lambda ni, no: torch.nn.init.xavier_uniform_(torch.empty(no,ni)), '#4C72B0'),
    ('Xavier normal',    lambda ni, no: torch.nn.init.xavier_normal_(torch.empty(no,ni)),  '#55A868'),
]

for strat_name, init_fn, col in strategies:
    x = x0.clone()
    var_hist = []
    for _ in range(n_layers):
        W = init_fn(n_units, n_units)
        x = torch.tanh(x @ W.T)
        var_hist.append(x.var().item())
    axes[0].plot(range(1, n_layers+1), var_hist, 'o-', lw=2, color=col,
                 label=strat_name, ms=6)

axes[0].set_xlabel('Layer depth'); axes[0].set_ylabel('Activation variance (tanh)')
axes[0].set_title('Xavier Init — Variance Preservation (tanh)')
axes[0].legend(); axes[0].set_yscale('symlog')
axes[0].axhline(0.1, color='grey', ls=':', lw=1)

# ── Manual Xavier forward pass ──
ni, no = 512, 256
W_xavier = torch.empty(no, ni)
torch.nn.init.xavier_uniform_(W_xavier)

axes[1].hist(W_xavier.numpy().flatten(), bins=60, color='#4C72B0', alpha=0.8, density=True)
expected_bound = np.sqrt(6 / (ni + no))
axes[1].axvline( expected_bound, color='#C44E52', ls='--', lw=2, label=f'+√(6/(ni+no))={expected_bound:.4f}')
axes[1].axvline(-expected_bound, color='#C44E52', ls='--', lw=2, label=f'-√(6/(ni+no))')
axes[1].set_xlabel('Weight value'); axes[1].set_ylabel('Density')
axes[1].set_title(f'Xavier Uniform: ni={ni}, no={no}'); axes[1].legend(fontsize=9)

# ── Variance formula validation ──
ni_vals = [64, 128, 256, 512, 1024]
stds_expected, stds_actual = [], []
for ni in ni_vals:
    W = torch.empty(128, ni); torch.nn.init.xavier_normal_(W)
    stds_actual.append(W.std().item())
    stds_expected.append(np.sqrt(2 / (ni + 128)))

axes[2].plot(ni_vals, stds_expected, 'o-', color='#4C72B0', lw=2, ms=8, label='Expected σ=√(2/(ni+no))')
axes[2].plot(ni_vals, stds_actual,  's--', color='#55A868', lw=2, ms=8, label='Actual σ (PyTorch)')
axes[2].set_xlabel('n_in'); axes[2].set_ylabel('Weight std')
axes[2].set_title('Xavier Normal: Formula vs PyTorch'); axes[2].legend()

plt.suptitle('Figure 9: Xavier / Glorot Initialization', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 11. He (Kaiming) Initialization — For ReLU Networks

### Why Xavier Fails for ReLU
Xavier was derived assuming **symmetric activations** (tanh, sigmoid) that preserve the mean of activations.

**ReLU kills all negative pre-activations** → half the neurons output 0:

$$\mathbb{E}[a] = \frac{1}{2} \cdot \mathbb{E}[z | z > 0] \neq 0$$

The effective variance is **halved** by ReLU. Xavier doesn't account for this.

Result: with Xavier + ReLU, variance decreases by $\approx 0.5$ per layer → signal collapses at depth.

---

### He Initialization Fix
He et al. (2015) — "Delving Deep into Rectifiers" — corrected for ReLU:

$$\sigma^2_W = \frac{2}{n_{\text{in}}}$$

The factor of 2 compensates for the half-zeroing by ReLU.

**He Normal:**
$$W \sim \mathcal{N}\left(0,\; \sqrt{\frac{2}{n_{\text{in}}}}\right)$$

**He Uniform:**
$$W \sim \mathcal{U}\left(-\sqrt{\frac{6}{n_{\text{in}}}},\; +\sqrt{\frac{6}{n_{\text{in}}}}\right)$$

### PyTorch
```python
nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
nn.init.kaiming_uniform_(layer.weight, mode='fan_in', nonlinearity='relu')
```

- `mode='fan_in'`: normalises for forward pass (more common)
- `mode='fan_out'`: normalises for backward pass
- `nonlinearity`: adjusts gain for other activations (leaky_relu, tanh, etc.)

### He vs Xavier — When to Use Which

| Activation | Init |
|-----------|------|
| ReLU, Leaky ReLU, ELU | **He (Kaiming)** |
| Tanh, Sigmoid | **Xavier (Glorot)** |
| Linear (no activation) | **Xavier** |
| Embeddings | **Small uniform** or Xavier |

**PyTorch's `nn.Linear` default is He Uniform** — good for ReLU networks.

#### Interview Questions: Initialization
> **Q: What is the symmetry breaking problem and how is it solved?**
> A: If all weights initialised to the same value, all neurons in a layer compute identical outputs and receive identical gradients — they stay identical forever, reducing the layer to effectively 1 neuron. Solved by random initialisation from a zero-mean distribution.

> **Q: Derive why He initialisation uses a factor of 2 vs Xavier.**
> A: Xavier assumes symmetric activations: $\text{Var}(\text{ReLU}(z)) = \text{Var}(z)$. But ReLU zeroes negative inputs, halving the variance: $\text{Var}(\text{ReLU}(z)) \approx \frac{1}{2}\text{Var}(z)$. To compensate, we need weight variance $\sigma^2_W = 2/n_{\text{in}}$ instead of $1/n_{\text{in}}$ — the factor of 2 exactly corrects for the halving.

> **Q: Should you initialise biases to zero or random?**
> A: Biases to zero — they don't have the symmetry problem since each neuron has its own bias. Random bias init adds unnecessary noise without benefit.


In [ ]:
# Figure 10: He vs Xavier — with ReLU activation
import torch, numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)

n_layers = 20
n_units  = 256
N_samp   = 1024
x0 = torch.randn(N_samp, n_units)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Variance through ReLU layers ──
for name, init_fn, col in [
    ('Xavier normal', lambda n: torch.nn.init.xavier_normal_(torch.empty(n,n)),  '#C44E52'),
    ('He (Kaiming)',  lambda n: torch.nn.init.kaiming_normal_(torch.empty(n,n)), '#55A868'),
    ('std=1.0',       lambda n: torch.randn(n,n),                                '#DD8452'),
]:
    x = x0.clone(); var_hist = []
    for _ in range(n_layers):
        W = init_fn(n_units)
        x = torch.relu(x @ W.T)
        var_hist.append(x.var().item())
    axes[0].plot(range(1, n_layers+1), var_hist, 'o-', lw=2, color=col,
                 label=name, ms=5, alpha=0.85)

axes[0].set_xlabel('Layer depth'); axes[0].set_ylabel('Activation variance (ReLU)')
axes[0].set_title('He Init Preserves Variance Through ReLU')
axes[0].legend(); axes[0].set_yscale('symlog')
axes[0].axhline(1.0, color='grey', ls=':', lw=1)

# ── He weight distribution ──
ni = 512
W_he  = torch.empty(256, ni); torch.nn.init.kaiming_normal_(W_he, nonlinearity='relu')
W_xav = torch.empty(256, ni); torch.nn.init.xavier_normal_(W_xav)

axes[1].hist(W_xav.numpy().flatten(), bins=60, color='#C44E52', alpha=0.6,
             density=True, label=f'Xavier σ={W_xav.std():.4f}')
axes[1].hist(W_he.numpy().flatten(),  bins=60, color='#55A868', alpha=0.6,
             density=True, label=f'He σ={W_he.std():.4f}')
axes[1].axvline(0, color='grey', ls='--', lw=1)
axes[1].set_xlabel('Weight value'); axes[1].set_ylabel('Density')
axes[1].set_title(f'He vs Xavier Distribution (ni={ni})')
# Expected std
he_expected   = np.sqrt(2/ni)
xav_expected  = np.sqrt(2/(ni+256))
axes[1].set_title(f'He σ_expected={he_expected:.4f} | Xavier σ_expected={xav_expected:.4f}', fontsize=9)
axes[1].legend()

# ── Gradient norms at each layer ──
def make_deep_relu(init_name, n_layers=10, n=128):
    layers = []
    for _ in range(n_layers):
        l = torch.nn.Linear(n, n, bias=False)
        if init_name == 'xavier':
            torch.nn.init.xavier_normal_(l.weight)
        elif init_name == 'he':
            torch.nn.init.kaiming_normal_(l.weight, nonlinearity='relu')
        elif init_name == 'zero':
            torch.nn.init.constant_(l.weight, 0.01)
        layers += [l, torch.nn.ReLU()]
    return torch.nn.Sequential(*layers, torch.nn.Linear(n, 1, bias=False))

for init_nm, col, lbl in [('xavier','#C44E52','Xavier'),('he','#55A868','He (Kaiming)')]:
    net = make_deep_relu(init_nm)
    x_in = torch.randn(64, 128); y_in = torch.randn(64, 1)
    torch.nn.MSELoss()(net(x_in), y_in).backward()
    gnorms = []
    for m in net.modules():
        if isinstance(m, torch.nn.Linear) and m.weight.grad is not None:
            gnorms.append(m.weight.grad.norm().item())
    axes[2].plot(range(len(gnorms)), gnorms, 'o-', lw=2, color=col, ms=6, label=lbl)

axes[2].set_xlabel('Layer (0=last)'); axes[2].set_ylabel('Gradient norm')
axes[2].set_title('Gradient Norms: Xavier vs He Init (ReLU net)')
axes[2].legend(); axes[2].set_yscale('log')

plt.suptitle('Figure 10: He (Kaiming) Initialization for ReLU', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 12. Orthogonal Initialization

### Intuition: Maximally Independent Neurons
Random normal init → neurons may be highly correlated → redundant representations.
**Orthogonal initialization** initialises weight matrices to be orthogonal — each row (neuron) is perpendicular to all others.

Benefits of orthogonal matrices:
1. **Preserve norm**: $\|Wx\|_2 = \|x\|_2$ — no signal amplification or collapse
2. **Eigenvalues = 1**: no exploding or vanishing gradients (in the linear case)
3. **Maximally diverse**: neurons start as uncorrelated as possible

### Math
Sample $W \in \mathbb{R}^{m \times n}$ (with $m \leq n$) as follows:
1. Sample $Z \in \mathbb{R}^{n \times n}$ from $\mathcal{N}(0, 1)$
2. Compute QR decomposition: $Z = QR$ (where $Q$ is orthogonal)
3. Let $W = Q[:m, :]$ — the first $m$ rows of $Q$

### When to Use Orthogonal Init
| Use case | Why |
|----------|-----|
| **RNNs** | Orthogonal hidden-to-hidden matrix → eigenvalues = 1 → no temporal gradient explosion |
| **Very deep MLPs** | Orthogonal matrices preserve singular values across depth |
| **Residual networks** | Combined with near-zero init for residual branch |

### PyTorch
```python
nn.init.orthogonal_(layer.weight, gain=1.0)
# gain: scale factor. gain=√2 for ReLU, 1.0 for linear
```

### Comparison Summary

| Init | Formula | Best For | PyTorch API |
|------|---------|---------|-------------|
| **Xavier Normal** | $\mathcal{N}(0, 2/(n_{in}+n_{out}))$ | Tanh, sigmoid | `xavier_normal_` |
| **Xavier Uniform** | $\mathcal{U}(-\sqrt{6/(n_{in}+n_{out})}, +)$ | Tanh, sigmoid | `xavier_uniform_` |
| **He Normal** | $\mathcal{N}(0, 2/n_{in})$ | ReLU variants | `kaiming_normal_` |
| **He Uniform** | $\mathcal{U}(-\sqrt{6/n_{in}}, +)$ | ReLU variants | `kaiming_uniform_` |
| **Orthogonal** | QR decomposition | RNNs, very deep | `orthogonal_` |
| **Zeros** | $0$ | Biases only | `zeros_` |


---

## 13. PyTorch Demo — Normalization + Initialization Together

Training a deep network under different combinations of normalization and initialization strategies.


In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt

torch.manual_seed(0)

# ── Dataset ──────────────────────────────────────────────────────
N = 512
X = torch.randn(N, 32)
y = (X[:, :16].sum(dim=1) > 0).long()   # simple linear classification

# ── Model factory ─────────────────────────────────────────────────
def make_model(norm_type, init_type, depth=8, width=64):
    layers = []
    n_in = 32
    for i in range(depth):
        lin = nn.Linear(n_in, width)
        # Initialization
        if init_type == 'zeros':
            nn.init.zeros_(lin.weight)
        elif init_type == 'xavier':
            nn.init.xavier_normal_(lin.weight)
        elif init_type == 'he':
            nn.init.kaiming_normal_(lin.weight, nonlinearity='relu')
        elif init_type == 'ortho':
            nn.init.orthogonal_(lin.weight)
        nn.init.zeros_(lin.bias)
        layers.append(lin)
        # Normalization
        if norm_type == 'bn':
            layers.append(nn.BatchNorm1d(width))
        elif norm_type == 'ln':
            layers.append(nn.LayerNorm(width))
        layers.append(nn.ReLU())
        n_in = width
    layers.append(nn.Linear(width, 2))
    return nn.Sequential(*layers)

def train_model(norm, init, epochs=200):
    model = make_model(norm, init)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    crit  = nn.CrossEntropyLoss()
    hist  = []
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        l = crit(model(X), y); l.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            acc = (model(X).argmax(1) == y).float().mean().item()
        hist.append(acc)
    return hist

configs = [
    ('none', 'zeros',  '#937860', 'No Norm + Zero Init'),
    ('none', 'he',     '#C44E52', 'No Norm + He Init'),
    ('bn',   'he',     '#4C72B0', 'BatchNorm + He Init'),
    ('ln',   'he',     '#55A868', 'LayerNorm + He Init'),
    ('bn',   'ortho',  '#8172B3', 'BatchNorm + Ortho Init'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

results = {}
for norm, init, col, label in configs:
    hist = train_model(norm, init)
    results[label] = hist
    axes[0].plot(hist, color=col, lw=2, label=label, alpha=0.85)
    print(f"{label:30s} | Final acc: {hist[-1]*100:.1f}%")

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Training Accuracy: Norm + Init Combinations')
axes[0].legend(fontsize=9); axes[0].set_ylim(0.3, 1.05)

# ── Bar chart final accuracy ──
labels = list(results.keys())
accs   = [results[l][-1]*100 for l in labels]
colors_bar = [c for *_, c, __ in configs]

bars = axes[1].bar(range(len(labels)), accs, color=[c for _,_,c,__ in configs], alpha=0.88)
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels([l.replace(' + ', '
+
') for l in labels], fontsize=8.5)
axes[1].set_ylabel('Final Accuracy (%)'); axes[1].set_title('Final Accuracy Comparison')
axes[1].set_ylim(50, 105)
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{acc:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Figure 11: Normalization + Initialization — Combined Effect', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 14. Master Interview Q&A Cheatsheet

### Level 1 — Beginner

> **Q: What is Batch Normalization and what problem does it solve?**
> A: BatchNorm normalises layer inputs to have zero mean and unit variance within each mini-batch. It solves Internal Covariate Shift — the shifting distribution of layer inputs as weights update — allowing faster training with larger learning rates and less sensitivity to initialization.

> **Q: Where in a network is BatchNorm typically applied?**
> A: After the linear/conv layer and before the activation: `Linear → BatchNorm → ReLU`. Some modern architectures use pre-norm (norm before the sub-layer) instead.

> **Q: Why do we need learnable γ and β in BatchNorm?**
> A: Without them, the network is forced to operate in the zero-mean, unit-variance regime forever. γ and β let the network undo the normalisation if that's optimal — e.g., learn γ=σ, β=μ to restore the original distribution.

> **Q: Why do transformers use LayerNorm instead of BatchNorm?**
> A: LayerNorm works at batch size 1 (needed for auto-regressive generation), handles variable-length sequences, and behaves identically at train and test time. BatchNorm requires large batches and breaks for single-sample inference.

---

### Level 2 — Mid-Level

> **Q: What happens to BatchNorm at batch size 1?**
> A: Catastrophic failure — with one sample, variance=0, causing division by zero (or ε). The running statistics accumulated during training are used at eval time, but training with batch=1 is broken. Solutions: LayerNorm, GroupNorm, or use a batch size ≥ 8.

> **Q: Explain He initialization and why it works for ReLU.**
> A: ReLU zeroes all negative pre-activations, halving the effective variance of each layer's output. He initialization doubles the normal Xavier variance ($2/n_{in}$ vs $1/n_{in}$) to exactly compensate, maintaining signal variance across depth.

> **Q: What is the difference between nn.BatchNorm1d train mode and eval mode?**
> A: Train mode: normalises using the current mini-batch statistics and updates running mean/var. Eval mode: normalises using the accumulated running statistics (stable population estimates). Forgetting `model.eval()` causes inconsistent normalisation at inference.

> **Q: GroupNorm vs BatchNorm for object detection — which would you choose and why?**
> A: GroupNorm. Object detection models use high-resolution images requiring much larger memory, limiting batch sizes to 1-2 per GPU. BatchNorm with batch=2 has unstable statistics (high variance of mini-batch estimates). GroupNorm is independent of batch size — it normalises within channel groups per sample, giving stable statistics at any batch size.

---

### Level 3 — Senior MLE / Staff Engineer

> **Q: A very deep (50-layer) ReLU network is suffering from vanishing gradients despite using He init. What would you try?**
> A: (1) Add BatchNorm or LayerNorm after each linear layer — normalisation is the primary fix for gradient stability at depth. (2) Add residual connections (skip connections) — bypasses gradient through layers directly. (3) Check activations with hooks — are there dead ReLU neurons? Switch to Leaky ReLU or GELU. (4) Re-verify He init is applied correctly — mode='fan_in', nonlinearity='relu'. (5) Consider orthogonal init for the initial layers. (6) Use gradient checkpointing to verify gradient magnitudes per layer.

> **Q: Explain how Pre-LayerNorm differs from Post-LayerNorm in transformers and why it matters.**
> A: Post-LN (original Transformer): LayerNorm applied after residual addition. Pre-LN (GPT-2, modern): LayerNorm applied before sublayer. Pre-LN enables training without learning rate warmup for deep transformers — the residual path is always clean, gradients flow unobstructed. Post-LN requires careful warmup and can collapse with aggressive LR. Pre-LN is now the overwhelming standard for deep transformers.

> **Q: How would you initialise the output layer of a language model (next-token prediction) for fast convergence?**
> A: (1) Small init for the output projection (std=0.02 as in GPT-2) to prevent initial logits from being too spread. (2) Init the output embedding matrix the same as the input embedding (weight tying) — reduces parameters and provides a good starting point. (3) For the final linear layer, some practitioners init to zero and add a learnable scale — "zero init trick" for residual branches. (4) biases to log(class_freq) if classes are imbalanced — prevents early training from wasting steps correcting the bias.
